### and write function to create graphs every 1000 steps cumulatively till n number of steps and 1 every 1000

### find out max num of workers and prefetch factor on h200 gpu to avoid OOMs
### quite a few variable were changed to suit the trial run on macbook check evry single one of var from lr to batch size to webloader and eval links for if they were modified like num worker and prefech factor batch siz efor webdataset and dataloader
### make sure to use bf16 or fp8 for training
### make sure global batch size is max the h200 can fit be it 4k or 8k or 512
### still need to make sure optimal lr for training throufh some sort of metrics or tiral and rrror not sre ideal rate what would be for my case of short run
### ask for best dropout rate is it 0.10 or 0.15 or 0.20
### find out highest batch size using pytorch func in claudes chat for validation and training size

In [ ]:
import psutil
import os


def print_cpu_memory():
    # System-wide memory
    mem = psutil.virtual_memory()
    print("\nCPU Memory:")
    print(f"  Total: {mem.total / 1024**3:.2f} GB")
    print(f"  Available: {mem.available / 1024**3:.2f} GB")
    print(f"  Used: {mem.used / 1024**3:.2f} GB")
    print(f"  Percentage: {mem.percent}%")

    # Current process memory
    process = psutil.Process(os.getpid())
    process_mem = process.memory_info().rss / 1024**3
    print(f"  This Process: {process_mem:.2f} GB")


print_cpu_memory()


CPU Memory:
  Total: 32.00 GB
  Available: 9.13 GB
  Used: 8.25 GB
  Percentage: 71.5%
  This Process: 0.03 GB


In [ ]:
def print_disk_usage(path="/"):
    """Check disk space for a given path"""
    disk = psutil.disk_usage(path)

    print("\nDisk Usage for '{path}':")
    print(f"  Total: {disk.total / 1024**3:.2f} GB")
    print(f"  Used: {disk.used / 1024**3:.2f} GB")
    print(f"  Free: {disk.free / 1024**3:.2f} GB")
    print(f"  Percentage: {disk.percent}%")


# Check root directory
print_disk_usage("/")

# Check specific paths
print_disk_usage("/home")
print_disk_usage(".")  # Current directory


Disk Usage for '/':
  Total: 926.35 GB
  Used: 11.42 GB
  Free: 151.23 GB
  Percentage: 7.0%

Disk Usage for '/home':
  Total: 0.00 GB
  Used: 0.00 GB
  Free: 0.00 GB
  Percentage: 0.0%

Disk Usage for '.':
  Total: 926.35 GB
  Used: 747.71 GB
  Free: 151.23 GB
  Percentage: 83.2%


In [ ]:
import torch

# Check initial state
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# Create a large tensor
x = torch.randn(10000, 10000, device="cuda")
print("\nAfter creating tensor:")
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")  # ~0.4 GB
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")  # Might be 2 GB

# Delete the tensor
del x
print("\nAfter deleting tensor:")
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")  # Back to ~0
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")  # Still 2 GB!

# Actually free the cached memory
torch.cuda.empty_cache()
print("\nAfter empty_cache():")
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")  # Still ~0
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")  # Now back to 0

In [ ]:
def find_max_eval_batch_size(model, device):
    """Binary search to find max batch size that fits in memory"""
    model.eval()
    batch_sizes = [512, 256, 128, 64, 32]

    for bs in batch_sizes:
        try:
            # Create dummy batch
            pixel_values = torch.randn(bs, 3, 256, 256, device=device)
            input_ids = torch.randint(0, 49408, (bs, 77), device=device)
            attention_mask = torch.ones(bs, 77, device=device)

            with torch.no_grad():
                _ = model(input_ids, attention_mask, pixel_values)

            print(f"✓ Batch size {bs} fits!")
            torch.cuda.empty_cache()
            return bs

        except torch.cuda.OutOfMemoryError:
            print(f"✗ Batch size {bs} OOM")
            torch.cuda.empty_cache()
            continue

    return 32  # Fallback

In [ ]:
def print_cpu_memory():
    # System-wide memory
    mem = psutil.virtual_memory()
    print("\nCPU Memory:")
    print(f"  Total: {mem.total / 1024**3:.2f} GB")
    print(f"  Available: {mem.available / 1024**3:.2f} GB")
    print(f"  Used: {mem.used / 1024**3:.2f} GB")
    print(f"  Percentage: {mem.percent}%")

    # Current process memory
    process = psutil.Process(os.getpid())
    process_mem = process.memory_info().rss / 1024**3
    print(f"  This Process: {process_mem:.2f} GB")


print_cpu_memory()

In [ ]:
import torch


def print_mps_memory():
    if torch.backends.mps.is_available():
        print("\nMPS Memory:")
        print(f"  Allocated: {torch.mps.current_allocated_memory() / 1024**3:.2f} GB")
        print(
            f"  Reserved (Driver): {torch.mps.driver_allocated_memory() / 1024**3:.2f} GB"
        )
    else:
        print("MPS not available")


# Example usage
if torch.backends.mps.is_available():
    x = torch.randn(1000, 1000, device="mps")
    print_mps_memory()

In [ ]:
def parse_training_log(log_path):
    opt_steps = []
    losses = []
    norms = []
    logit_scales = []

    with open(log_path, "r") as f:
        for line in f:
            parts = line.strip().split(" | ")
            kv = {}
            for p in parts:
                if "=" in p:
                    k, v = p.split("=")
                    kv[k.strip()] = v.strip()

            # required fields
            opt_steps.append(int(kv["opt_step"]))
            losses.append(float(kv["loss"]))
            norms.append(float(kv["norm"]))
            logit_scales.append(float(kv["logit_scale"]))

    return opt_steps, losses, norms, logit_scales


opt_steps, losses, norms, logit_scales = parse_training_log("logs/training.log")